# GNNs (GCN / GraphSAGE / GAT)  
 

In [1]:
import torch, math
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import torch_geometric
from torch_geometric.utils import add_self_loops, to_undirected
from torch_geometric.utils import scatter

device = "cuda" if torch.cuda.is_available() else "cpu"


## 1) Message Passing recap

A generic message passing layer:


$h_v^{(l+1)} = \phi\left(h_v^{(l)},\; \square_{u \in \mathcal{N}(v)} \psi(h_v^{(l)}, h_u^{(l)}, e_{uv})\right)$

- **GCN**: normalized sum aggregation  
- **GraphSAGE**: learnable aggregation (we use mean)  
- **GAT**: attention-weighted aggregation  


## 2) Visualizers

In [2]:
def visualize_graph_2d(xy, edge_index, title="Graph (2D)", max_edges=800):
    xy = xy.detach().cpu()
    ei = edge_index.detach().cpu()
    plt.figure(figsize=(5,5))
    plt.scatter(xy[:,0], xy[:,1], s=15, alpha=0.8)
    E = ei.size(1)
    for j in range(min(E, max_edges)):
        a = ei[0,j].item()
        b = ei[1,j].item()
        plt.plot([xy[a,0], xy[b,0]], [xy[a,1], xy[b,1]], linewidth=0.5, alpha=0.3)
    plt.title(title)
    plt.axis("equal")
    plt.show()

def visualize_points_3d(pos, edge_index=None, title="Point cloud", max_edges=1500):
    pos = pos.detach().cpu()
    fig = plt.figure(figsize=(6,5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(pos[:,0], pos[:,1], pos[:,2], s=10, alpha=0.85)
    if edge_index is not None:
        ei = edge_index.detach().cpu()
        E = ei.size(1)
        for j in range(min(E, max_edges)):
            a = ei[0,j].item()
            b = ei[1,j].item()
            p1 = pos[a]; p2 = pos[b]
            ax.plot([p1[0],p2[0]],[p1[1],p2[1]],[p1[2],p2[2]], linewidth=0.3, alpha=0.2)
    ax.set_title(title)
    ax.set_box_aspect([1,1,1])
    plt.tight_layout()
    plt.show()


## 3) kNN graph builder  

Returns `edge_index` with shape `[2, N*k]` (directed).


In [3]:
def knn_graph_torch(pos: torch.Tensor, k: int, loop: bool = False) -> torch.Tensor:
    dist = torch.cdist(pos, pos)  # Eucl dist point-to-point [N, N]
    if not loop: #loop=true => self-loop
        dist.fill_diagonal_(float("inf"))
    knn = dist.topk(k, largest=False).indices  # prende i k indici con distanza minima [N, k]
    row = torch.arange(pos.size(0), device=pos.device).view(-1,1).repeat(1,k).reshape(-1) #nodi che ricevono i messaggi. se k=2 [0011223344]
    col = knn.reshape(-1) #nodi che mandano i messaggi [12 (nodi connessi a 0) 03 (nodi connessi a 1) ... ]
    return torch.stack([row, col], dim=0) #[2, N*k]


## 4) Node classification on Cora (PyG layers)
Cora è un dataset classico per node classification su grafi.

Contiene:
- Paper scientifici
- Citazioni tra paper
- Categorie tematiche (classi)

Ogni nodo = un paper
Ogni edge = una citazione

Cora ha:
- 2708 nodi (paper)
- 5429 archi (citazioni, non diretti in PyG)
- 1433 feature per nodo
- 7 classi

Le 7 classi rappresentano aree di ricerca (es. neural networks, rule learning, etc.).

Cora ha uno split fisso:
- 140 nodi di training (20 per classe)
- 500 validation
- 1000 test

In [4]:
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

cora = Planetoid(root="data/Planetoid", name="Cora")
data = cora[0].to(device)
print(cora, data)

Processing...


Cora() Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Done!


- Node Classification

In [5]:
class GCNNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hid_dim)
        self.conv2 = GCNConv(hid_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class SAGENet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hid_dim)
        self.conv2 = SAGEConv(hid_dim, out_dim)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

class GATNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, heads=8, dropout=0.6):
        super().__init__()
        self.gat1 = GATConv(in_dim, hid_dim, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hid_dim*heads, out_dim, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.gat2(x, edge_index)

In [6]:
@torch.no_grad()
def acc(logits, y):
    return (logits.argmax(dim=-1) == y).float().mean().item()

def train_cora(model, data, lr=0.01, wd=5e-4, epochs=200):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_test = 0.0, 0.0
    for ep in range(1, epochs+1):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

        model.eval()
        out = model(data.x, data.edge_index)
        val = acc(out[data.val_mask], data.y[data.val_mask])
        test = acc(out[data.test_mask], data.y[data.test_mask])
        if val > best_val:
            best_val, best_test = val, test
        if ep % 50 == 0 or ep == 1:
            tr = acc(out[data.train_mask], data.y[data.train_mask])
            print(f"ep {ep:03d} | loss {loss.item():.3f} | train {tr:.3f} | val {val:.3f} | test {test:.3f}")
    print("best val:", best_val, "| test@best:", best_test)

print("--- GCN ---")
train_cora(GCNNet(cora.num_features, 16, cora.num_classes), data, lr=0.01, wd=5e-4)

print("\n--- SAGE ---")
train_cora(SAGENet(cora.num_features, 32, cora.num_classes), data, lr=0.01, wd=5e-4)

print("\n--- GAT ---")
train_cora(GATNet(cora.num_features, 8, cora.num_classes, heads=8), data, lr=0.005, wd=5e-4)


--- GCN ---
ep 001 | loss 1.951 | train 0.521 | val 0.344 | test 0.354
ep 050 | loss 0.060 | train 1.000 | val 0.764 | test 0.781
ep 100 | loss 0.046 | train 1.000 | val 0.762 | test 0.787
ep 150 | loss 0.035 | train 1.000 | val 0.772 | test 0.799
ep 200 | loss 0.036 | train 1.000 | val 0.766 | test 0.801
best val: 0.777999997138977 | test@best: 0.7829999923706055

--- SAGE ---
ep 001 | loss 1.945 | train 0.829 | val 0.412 | test 0.402
ep 050 | loss 0.009 | train 1.000 | val 0.774 | test 0.794
ep 100 | loss 0.007 | train 1.000 | val 0.764 | test 0.797
ep 150 | loss 0.006 | train 1.000 | val 0.770 | test 0.799
ep 200 | loss 0.008 | train 1.000 | val 0.784 | test 0.806
best val: 0.7839999794960022 | test@best: 0.8050000071525574

--- GAT ---
ep 001 | loss 2.022 | train 0.336 | val 0.258 | test 0.232
ep 050 | loss 0.531 | train 1.000 | val 0.772 | test 0.804
ep 100 | loss 0.500 | train 1.000 | val 0.784 | test 0.814
ep 150 | loss 0.442 | train 1.000 | val 0.772 | test 0.804
ep 200 | loss 